In [ ]:
import sys
import os
import pandas as pd

# ---------------- Correctly set path to hybridtablerag ----------------
notebook_dir = os.getcwd()  # e.g., D:\Nikita\AI ML Engineer\s2s dynamics\HybridTableRag\notebooks
hybridtablerag_path = os.path.abspath(os.path.join(notebook_dir, "..", "hybridtablerag"))

if hybridtablerag_path not in sys.path:
    sys.path.append(hybridtablerag_path)
    print("Added to sys.path:", hybridtablerag_path)

# ---------------- Verify metadata folder ----------------
metadata_path = os.path.join(hybridtablerag_path, "metadata")
print("Contents of metadata folder:", os.listdir(metadata_path))

# ---------------- Import your cleaner ----------------
from metadata.schema_profiler import clean_and_profile

# ---------------- Load CSV ----------------
data_path = os.path.join(os.path.abspath(os.path.join(notebook_dir, "..", "data", "synthetic")),
                         "professional_incident_tickets.csv")
df = pd.read_csv(data_path, header=[0,1])  # multi-row header

# ---------------- Clean and profile ----------------
result = clean_and_profile(df)
cleaned_df = result['cleaned_df']
log = result['log']

# ---------------- Inspect results ----------------
print("\nColumns after cleaning:")
print(cleaned_df.columns.tolist())

display(cleaned_df.head(10))

Added to sys.path: d:\Nikita\AI ML Engineer\s2s dynamics\HybridTableRag\hybridtablerag
Contents of metadata folder: ['relationship_detector.py', 'schema_profiler.py', '__init__.py', '__pycache__']

Columns after cleaning:
['ticket_ticket_id', 'ticket_created_date', 'ticket_resolved_date', 'ticket_priority', 'ticket_impacted_departments', 'customer_name', 'customer_email', 'resolution_description', 'resolution_affected_systems_system', 'resolution_affected_systems_impact', 'resolution_actions_taken_performed_by', 'resolution_actions_taken_step']


,ticket_ticket_id,ticket_created_date,ticket_resolved_date,ticket_priority,ticket_impacted_departments,customer_name,customer_email,resolution_description,resolution_affected_systems_system,resolution_affected_systems_impact,resolution_actions_taken_performed_by,resolution_actions_taken_step
0,TCKT-1000,2024-05-03,2024-05-29,Medium,Finance;IT;Operations,Gabriella Davis,trevinomichele@example.net,Of woman energy trade suddenly process you. Pe...,Payroll System,Full,Victoria Brown; Cassandra Jones,Involve gun democratic several including wish....
1,TCKT-1001,2025-03-31,2025-04-28,Low,HR;IT,Tiffany Howell,None,Machine bar minute. Traditional us production.,Payroll System,Partial,Connie Johnson; Daniel Rangel,True car site.; Increase politics half someone...
2,TCKT-1002,2024-07-10,2024-08-01,Medium,Sales;Operations,Brandon Maxwell,joescott@example.com,Brother build scene interview. Later idea assu...,Database; Email Server,Full; Partial,Amanda Gray,Read forget issue Democrat kitchen bring.
3,TCKT-1003,2025-12-22,2026-01-09,High,IT,Jennifer Baker,robertsjaime@example.net,Follow such chance scene anything describe. Ow...,Payroll System,Full,Tyler Patel; Amy Reyes,Little century million exactly everybody last ...
4,TCKT-1004,2024-11-23,2024-11-29,Medium,Marketing;HR,April Padilla,michelle09@example.com,Page window nature color resource order what. ...,Web Portal; Network Switch; Payroll System,Full; Partial; Partial,Sharon Byrd; Timothy Frederick; Colton Hancock,Threat during cost stay land color go.; Yes ar...
5,TCKT-1005,2024-04-16,2024-05-03,High,Finance;IT,Timothy Allen,lori24@example.org,Lot improve offer scientist. Dark office envir...,Database; Database,Partial; Partial,Joshua Bryant; Alison Armstrong,Memory program day government early race certa...
6,TCKT-1006,2024-08-12,2025-01-05,Medium,Finance;IT;Sales,Destiny Malone,rebecca92@example.com,Staff rock here order. Too else seem know remain.,Email Server,Partial,Courtney Mercado; Nicole Smith,Tree seven page believe doctor indeed down.; S...
7,TCKT-1007,2025-10-04,2025-10-10,Low,Support;Sales;IT,Donald Vaughan,dianasnyder@example.net,Idea his everything agency weight Mrs characte...,Database; Payroll System; CRM,Partial; Full; Partial,Chelsea White,Choice this probably accept.
8,TCKT-1008,2026-01-05,2026-01-27,Medium,Finance,Michael Garcia,eric26@example.org,Try professional include all technology. Artic...,Network Switch,Partial,William Powers; Christopher Carr; Eric Le,Experience major environmental.; Huge likely f...
9,TCKT-1009,2025-07-14,2025-07-20,High,Support;Operations,Steven Johnson,hoffmanjoshua@example.com,Federal operation serious according drug small...,Email Server,Partial,Alisha Jackson; Charles Hamilton,Task physical in heart response.; Image sign w...


In [ ]:

# Detect number of header rows automatically
# (peek at the file to decide whether it needs header=[0,1])
def _detect_header_rows(path: str, max_check: int = 2) -> list[int] | int:
    """
    Return header=[0,1] if the first two rows look like a two-level header
    (i.e. row 1 values are all subsets of row 0 column names), else return 0.
    """
    peek = pd.read_csv(path, nrows=1, header=None)
    row0 = peek.iloc[0].astype(str).tolist()
    # Simple heuristic: if first row has repeated values it's a group header
    if len(set(row0)) < len(row0):           # duplicates → multi-row header
        return [0, 1]
    return 0

hdr = _detect_header_rows(data_path)
print(f"Using header={hdr}")
df_raw = pd.read_csv(data_path, header=hdr)
print(f"Loaded:  {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

# ── Cell 3 : clean & profile ────────────────────────────────
result     = clean_and_profile(df_raw)
cleaned_df = result["cleaned_df"]
profile    = result["profile"]
log        = result["log"]

print(f"\n Cleaned:  {cleaned_df.shape[0]} rows × {cleaned_df.shape[1]} columns")
print(f"   Dropped duplicate content columns:  "
      f"{sum(1 for l in log if 'content-duplicate' in l)}")
print(f"   Date columns normalised:            "
      f"{sum(1 for l in log if 'Normalised dates' in l)}")
print(f"   JSON columns flattened:             "
      f"{sum(1 for l in log if 'Flattened' in l and '->' in l)}")

# ── Cell 4 : full log ───────────────────────────────────────
print("\n── Cleaning log ──────────────────────────────────────")
for line in log:
    prefix = (
        "   " if "dropped" in line.lower()  else
        "   " if "normalised" in line.lower() else
        "   " if "flattened" in line.lower()  else
        "   " if "numeric"   in line.lower()  else
        "     "
    )
    print(prefix + line)

# ── Cell 5 : column profile table ───────────────────────────
profile_rows = []
for col, stats in profile["columns"].items():
    profile_rows.append({
        "column":        col,
        "dtype":         stats["dtype"],
        "nulls":         stats["num_nulls"],
        "% null":        f"{stats['pct_null']:.1f}",
        "unique values": stats["num_unique"],
        "samples":       " | ".join(str(v) for v in stats["sample_values"][:3]),
    })

profile_df = pd.DataFrame(profile_rows).set_index("column")

# Style: highlight high null % in red
def _highlight_nulls(val):
    try:
        pct = float(str(val).replace("%", ""))
        if pct >= 50:  return "background-color:#3d1a1a; color:#f87171"
        if pct >= 20:  return "background-color:#3d300a; color:#f59e0b"
    except ValueError:
        pass
    return ""

display(
    profile_df.style
    .applymap(_highlight_nulls, subset=["% null"])
    .set_properties(**{"font-size": "12px"})
    .set_table_styles([{"selector": "th", "props": [("font-size", "12px")]}])
)

# ── Cell 6 : cleaned data ───────────────────────────────────
pd.set_option("display.max_columns",  None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width",        200)

print("\n── Cleaned DataFrame ─────────────────────────────────")
print("Columns:", cleaned_df.columns.tolist())
display(cleaned_df.head(10))
